# UCSC 34-Class Wildlife + Banana Slug (358) + Campus Trees + UCSC Bugs: Cloud-Only Training

Open this notebook in Google Colab, choose **Runtime -> Change runtime type -> T4 GPU** (or G4/L4/A100), then **Runtime -> Run all**.

- **34 classes:** 20 animals (`M01`–`M10`, `B01`–`B10`) × 1,000 + Banana slug (`I01`, genus `Ariolimax`) × **358** + 7 campus trees (`T01`–`T07`) × 1,000 + 6 UCSC bugs (`G01`–`G06`) × 1,000 = **33,358 real CC0/CC-BY images**.
- **Unique-observation limit removed:** Multiple distinct photos (`sha256` + `phash` deduplicated) from the same observation are accepted to hit targets cleanly, while splits remain grouped by `observer_id` so no photographer/observation leaks across splits.
- **Multi-epoch BioCLIP 2 ViT-L/14 training:** Up to 30 head epochs + 10 fine-tuning epochs of the last 2 visual transformer blocks + `ln_post` + `proj`, with early stopping, temperature scaling on a dedicated calibration split, and full 34-class test evaluation.
- **Cloud-only:** Refuses to run on macOS or without a hosted Colab CUDA GPU.

In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.platform != 'linux' or not (os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION')):
    raise RuntimeError('Run this notebook in hosted Google Colab, not on your computer.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun.')
print('Remote GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'open_clip_torch>=2.26.1', 'ImageHash>=4.3.1',
                'scikit-learn>=1.5.0', 'huggingface-hub>=0.24.0'], check=True)
print('Dependency setup complete.')

In [ ]:
RUN_NAME = 'ucsc34_bugs_slug358_bioclip_v4'
DEFAULT_IMAGES_PER_CLASS = 1000
BANANA_SLUG_IMAGES = 358
HEAD_EPOCHS = 30
FINETUNE_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
BATCH_SIZE = 16
SEED = 42
PERSIST_TO_DRIVE = True

if not RUN_NAME.replace('_', '').replace('-', '').isalnum():
    raise ValueError('Use letters, digits, underscores, and hyphens in RUN_NAME.')
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/DEVFEST_training') / RUN_NAME
else:
    ROOT = Path('/content/DEVFEST_training') / RUN_NAME
print('Remote dataset/checkpoint folder:', ROOT)
import shutil
space = shutil.disk_usage('/content')
print(f'Colab working disk free: {space.free / 2**30:.1f} GiB')
if space.free < 15 * 2**30:
    raise RuntimeError('Less than 15 GiB free on Colab working disk.')


In [ ]:
import json
CLASSES = [
    {"class_id": "M01", "index": 0, "common_name": "Black-tailed deer", "scientific_name": "Odocoileus hemionus", "subspecies_filter": "Odocoileus hemionus columbianus", "group": "large_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M02", "index": 1, "common_name": "Coyote", "scientific_name": "Canis latrans", "group": "large_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M03", "index": 2, "common_name": "Bobcat", "scientific_name": "Lynx rufus", "group": "large_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M04", "index": 3, "common_name": "Gray fox", "scientific_name": "Urocyon cinereoargenteus", "group": "large_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M05", "index": 4, "common_name": "Northern raccoon", "scientific_name": "Procyon lotor", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M06", "index": 5, "common_name": "Striped skunk", "scientific_name": "Mephitis mephitis", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M07", "index": 6, "common_name": "California ground squirrel", "scientific_name": "Otospermophilus beecheyi", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M08", "index": 7, "common_name": "Eastern gray squirrel", "scientific_name": "Sciurus carolinensis", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M09", "index": 8, "common_name": "Western gray squirrel", "scientific_name": "Sciurus griseus", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "M10", "index": 9, "common_name": "Brush rabbit", "scientific_name": "Sylvilagus bachmani", "group": "small_mammal", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B01", "index": 10, "common_name": "Wild turkey", "scientific_name": "Meleagris gallopavo", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B02", "index": 11, "common_name": "California quail", "scientific_name": "Callipepla californica", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B03", "index": 12, "common_name": "Steller's jay", "scientific_name": "Cyanocitta stelleri", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B04", "index": 13, "common_name": "California scrub-jay", "scientific_name": "Aphelocoma californica", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B05", "index": 14, "common_name": "Common raven", "scientific_name": "Corvus corax", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B06", "index": 15, "common_name": "Acorn woodpecker", "scientific_name": "Melanerpes formicivorus", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B07", "index": 16, "common_name": "Anna's hummingbird", "scientific_name": "Calypte anna", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B08", "index": 17, "common_name": "American robin", "scientific_name": "Turdus migratorius", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B09", "index": 18, "common_name": "Dark-eyed junco", "scientific_name": "Junco hyemalis", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "B10", "index": 19, "common_name": "Spotted towhee", "scientific_name": "Pipilo maculatus", "group": "bird", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "I01", "index": 20, "common_name": "Banana slug (genus)", "scientific_name": "Ariolimax", "group": "gastropod", "taxonomic_rank": "genus", "target_count": BANANA_SLUG_IMAGES},
    {"class_id": "T01", "index": 21, "common_name": "Coast redwood", "scientific_name": "Sequoia sempervirens", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T02", "index": 22, "common_name": "Douglas-fir", "scientific_name": "Pseudotsuga menziesii", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T03", "index": 23, "common_name": "Pacific madrone", "scientific_name": "Arbutus menziesii", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T04", "index": 24, "common_name": "Coast live oak", "scientific_name": "Quercus agrifolia", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T05", "index": 25, "common_name": "Tanoak", "scientific_name": "Notholithocarpus densiflorus", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T06", "index": 26, "common_name": "California bay", "scientific_name": "Umbellularia californica", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "T07", "index": 27, "common_name": "Bigleaf maple", "scientific_name": "Acer macrophyllum", "group": "tree", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G01", "index": 28, "common_name": "Monarch butterfly", "scientific_name": "Danaus plexippus", "group": "insect", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G02", "index": 29, "common_name": "Western tiger swallowtail", "scientific_name": "Papilio rutulus", "group": "insect", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G03", "index": 30, "common_name": "Yellow-faced bumble bee", "scientific_name": "Bombus vosnesenskii", "group": "insect", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G04", "index": 31, "common_name": "Yellow-spotted millipede", "scientific_name": "Harpaphe haydeniana", "group": "arthropod", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G05", "index": 32, "common_name": "Convergent lady beetle", "scientific_name": "Hippodamia convergens", "group": "insect", "target_count": DEFAULT_IMAGES_PER_CLASS},
    {"class_id": "G06", "index": 33, "common_name": "Western yellowjacket", "scientific_name": "Vespula pensylvanica", "group": "insect", "target_count": DEFAULT_IMAGES_PER_CLASS},
]
assert len(CLASSES) == 34
assert [s['index'] for s in CLASSES] == list(range(34))
CONFIG = dict(run_name=RUN_NAME, root=str(ROOT), classes=CLASSES,
              seed=SEED, head_epochs=HEAD_EPOCHS, finetune_epochs=FINETUNE_EPOCHS,
              patience=EARLY_STOPPING_PATIENCE, batch_size=BATCH_SIZE)
for item in CLASSES:
    print(f"{item['class_id']} {item['common_name']} ({item['scientific_name']}) -> target={item['target_count']}")
Path('/content/ucsc_v4_config.json').write_text(json.dumps(CONFIG, indent=2))


In [ ]:
%%writefile /content/train_cloud_v4.py
"""UCSC 34-class cloud-only BioCLIP 2 training (358 banana slug, 1000 x 33 others including UCSC bugs)."""
from pathlib import Path
import os
import sys


def require_cloud_gpu():
    if sys.platform != 'linux' or not (os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION')):
        raise RuntimeError('Cloud-only runner: execute inside a hosted Google Colab GPU runtime.')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('No CUDA GPU. Select Runtime > Change runtime type > T4 GPU, then rerun.')


def main(config):
    require_cloud_gpu()
    import collections
    import hashlib
    import io
    import json
    import random
    import shutil
    import subprocess
    import time
    import numpy as np
    import requests
    import imagehash
    import torch
    import torch.nn.functional as F
    from PIL import Image, ImageOps
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    from torch.utils.data import DataLoader, Dataset, TensorDataset
    import open_clip
    from huggingface_hub import HfApi, snapshot_download

    seed = config['seed']
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    root = Path(config['root'])
    root.mkdir(parents=True, exist_ok=True)
    classes = config['classes']
    nclass = len(classes)
    run = root / 'run'
    run.mkdir(exist_ok=True)

    def write_json(path, value):
        path = Path(path)
        temp = path.with_suffix(path.suffix + '.tmp')
        temp.write_text(json.dumps(value, indent=2))
        temp.replace(path)

    def append_json(path, value):
        with Path(path).open('a') as f:
            f.write(json.dumps(value) + '\n')

    old = root / 'config.json'
    if old.exists() and json.loads(old.read_text()) != config:
        raise RuntimeError('Configuration changed. Choose a new RUN_NAME to avoid mixing experiments.')
    write_json(old, config)
    (run / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    (run / 'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True))
    print('GPU:', torch.cuda.get_device_name(0), flush=True)

    session = requests.Session()
    session.headers['User-Agent'] = 'UCSC-DEVFEST-Wildlife-Research/4.0'
    last_api = 0.0

    def api(endpoint, params):
        nonlocal last_api
        for attempt in range(6):
            time.sleep(max(0, 1.1 - (time.monotonic() - last_api)))
            last_api = time.monotonic()
            try:
                r = session.get('https://api.inaturalist.org/v1/' + endpoint, params=params, timeout=45)
                if r.status_code == 429 or r.status_code >= 500:
                    time.sleep(min(120, max(2 ** attempt, float(r.headers.get('Retry-After', 0)))))
                    continue
                r.raise_for_status()
                return r.json()
            except requests.RequestException:
                if attempt == 5:
                    raise
                time.sleep(2 ** attempt)
        raise RuntimeError('iNaturalist API unavailable after retries; rerun to resume.')

    class BKTree:
        def __init__(self):
            self.root = None

        def near(self, value, radius=4):
            pending = [self.root] if self.root else []
            while pending:
                old_value, children = pending.pop()
                distance = (value ^ old_value).bit_count()
                if distance <= radius:
                    return True
                pending.extend(v for k, v in children.items() if abs(k - distance) <= radius)
            return False

        def add(self, value):
            if self.root is None:
                self.root = (value, {})
                return
            node = self.root
            while True:
                distance = (value ^ node[0]).bit_count()
                if not distance:
                    return
                if distance not in node[1]:
                    node[1][distance] = (value, {})
                    return
                node = node[1][distance]

    accepted = root / 'accepted.jsonl'
    rows = [json.loads(x) for x in accepted.read_text().splitlines()] if accepted.exists() else []
    hashes, photos = set(), set()
    tree = BKTree()
    counts = collections.Counter()
    for row in rows:
        p = root / row['path']
        if not p.exists() or hashlib.sha256(p.read_bytes()).hexdigest() != row['sha256']:
            raise RuntimeError(f'Saved dataset file missing/corrupt: {p}. Restore it or use a new RUN_NAME.')
        hashes.add(row['sha256'])
        photos.add(row['photo_id'])
        tree.add(int(row['phash'], 16))
        counts[row['class_id']] += 1
    media_bytes = 0
    media_start = time.monotonic()

    def fetch_photo(photo):
        nonlocal media_bytes
        url = photo['url'].replace('/square.', '/large.').replace('/small.', '/large.')
        for attempt in range(3):
            try:
                with session.get(url, timeout=45, stream=True) as response:
                    response.raise_for_status()
                    chunks, size = [], 0
                    for chunk in response.iter_content(65536):
                        size += len(chunk)
                        media_bytes += len(chunk)
                        time.sleep(max(0, media_bytes / 1_000_000 - (time.monotonic() - media_start)))
                        if media_bytes > 24_000_000_000:
                            raise RuntimeError('24 GB per-day media limit reached; resume later.')
                        if size > 20_000_000:
                            raise ValueError('Oversized source image')
                        chunks.append(chunk)
                    content = b''.join(chunks)
                with Image.open(io.BytesIO(content)) as img:
                    rgb = ImageOps.exif_transpose(img).convert('RGB')
                    rgb.load()
                    if min(rgb.size) < 224:
                        return None
                    return content, hashlib.sha256(content).hexdigest(), str(imagehash.phash(rgb)), rgb.size
            except (requests.RequestException, OSError, ValueError) as exc:
                if attempt == 2:
                    append_json(root / 'rejected.jsonl', {'photo_id': photo['id'], 'reason': str(exc)})
                    return None
                time.sleep(2 ** attempt)

    resolved = []
    availability = {}
    for index, sp in enumerate(classes):
        query = sp.get('subspecies_filter') or sp['scientific_name']
        candidates = api('taxa', {'q': query, 'per_page': 30})['results']
        exact = [t for t in candidates if t['name'].lower() == query.lower() and t.get('is_active', True)]
        if len(exact) != 1:
            raise RuntimeError(f'Cannot resolve exact taxon {query}; inspect taxonomy before continuing.')
        taxon_id = exact[0]['id']
        resolved.append({**sp, 'index': index, 'resolved_taxon_id': taxon_id, 'query_name': query})
        metadata = api('observations', dict(taxon_id=taxon_id, quality_grade='research',
                       photos='true', photo_license='cc0,cc-by', per_page=1, verifiable='true'))
        availability[sp['class_id']] = metadata['total_results']
    write_json(root / 'availability.json', availability)
    print('Metadata availability preflight:', availability, flush=True)

    for index, sp in enumerate(resolved):
        taxon_id = sp['resolved_taxon_id']
        target = int(sp['target_count'])
        if counts[sp['class_id']] >= target:
            print(sp['common_name'], counts[sp['class_id']], '(resumed)', flush=True)
            continue
        # Ignore unique observation restriction: accept multiple distinct photos per observation.
        for place in [14, None]:
            for page in range(1, 51):
                if counts[sp['class_id']] >= target:
                    break
                params = dict(taxon_id=taxon_id, quality_grade='research', photos='true',
                              photo_license='cc0,cc-by', per_page=200, page=page,
                              order_by='id', order='desc', verifiable='true')
                if place:
                    params['place_id'] = place
                observations_page = api('observations', params)['results']
                if not observations_page:
                    break
                for obs in observations_page:
                    oid = str(obs['id'])
                    actual_taxon = obs.get('taxon') or {}
                    if taxon_id not in [actual_taxon.get('id'), *actual_taxon.get('ancestor_ids', [])]:
                        continue
                    if not (obs.get('user') or {}).get('id'):
                        continue
                    for photo in obs.get('photos', []):
                        pid = str(photo['id'])
                        if pid in photos or photo.get('license_code') not in ('cc0', 'cc-by'):
                            continue
                        result = fetch_photo(photo)
                        if result is None:
                            continue
                        content, sha, phash, size = result
                        if sha in hashes or tree.near(int(phash, 16)):
                            append_json(root / 'rejected.jsonl', {'photo_id': pid, 'reason': 'duplicate_or_near_duplicate'})
                            continue
                        path = Path('images') / sp['class_id'] / (pid + '.jpg')
                        (root / path).parent.mkdir(parents=True, exist_ok=True)
                        (root / path).write_bytes(content)
                        row = dict(class_id=sp['class_id'], label=index, path=str(path),
                                   observation_id=oid, observer_id=str(obs['user']['id']), photo_id=pid,
                                   sha256=sha, phash=phash, width=size[0], height=size[1],
                                   taxon_id=actual_taxon.get('id'), actual_name=actual_taxon.get('name'),
                                   query_taxon_id=taxon_id, license=photo['license_code'],
                                   attribution=photo.get('attribution') or obs['user'].get('login'),
                                   source_url=f'https://www.inaturalist.org/photos/{pid}',
                                   image_url=photo['url'].replace('/square.', '/large.'),
                                   observed_on=obs.get('observed_on'),
                                   retrieval_scope='California' if place else 'global')
                        append_json(accepted, row)
                        rows.append(row)
                        photos.add(pid)
                        hashes.add(sha)
                        tree.add(int(phash, 16))
                        counts[sp['class_id']] += 1
                        if counts[sp['class_id']] % 100 == 0:
                            print(f"{sp['common_name']}: {counts[sp['class_id']]}/{target}", flush=True)
                        if counts[sp['class_id']] >= target:
                            break
                    if counts[sp['class_id']] >= target:
                        break
                if len(observations_page) < 200:
                    break
            if counts[sp['class_id']] >= target:
                break
        print(f"CLASS COMPLETE: {sp['common_name']}: {counts[sp['class_id']]}/{target}", flush=True)
        write_json(root / 'counts.json', dict(counts))
    write_json(run / 'classes.json', resolved)

    shortages = {s['common_name']: counts[s['class_id']] for s in classes if counts[s['class_id']] < int(s['target_count'])}
    if shortages:
        write_json(root / 'shortfalls.json', shortages)
        raise RuntimeError(f'Insufficient real licensed images; no training started. Shortfalls: {shortages}')

    labels = np.array([r['label'] for r in rows])
    groups = np.array([r['observer_id'] for r in rows])
    fold = np.full(len(rows), -1)
    splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    for k, (_, ids) in enumerate(splitter.split(np.zeros(len(rows)), labels, groups)):
        fold[ids] = k
    split_names = np.array(['test' if f < 2 else 'calibration' if f == 2 else 'validation' if f == 3 else 'train' for f in fold])
    for row, name in zip(rows, split_names):
        row['split'] = str(name)
    split_sets = {s: set(groups[split_names == s]) for s in set(split_names)}
    for s, a in split_sets.items():
        for t, b in split_sets.items():
            if s != t:
                assert not (a & b), 'Photographer leakage'
    split_counts = {s: np.bincount(labels[split_names == s], minlength=nclass).tolist() for s in split_sets}
    write_json(run / 'split_counts.json', split_counts)
    manifest_text = '\n'.join(json.dumps(r, sort_keys=True) for r in rows) + '\n'
    manifest_hash = hashlib.sha256(manifest_text.encode()).hexdigest()
    (run / 'manifest.jsonl').write_text(manifest_text)
    write_json(run / 'dataset_identity.json', {'sha256': manifest_hash, 'split_policy': 'photographer-grouped 60/10/10/20'})
    print('Validated actual images and leakage-free splits:', split_counts, flush=True)

    cache = Path('/content/ucsc_image_cache') / config['run_name']
    for row in rows:
        dest = cache / row['path']
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(root / row['path'], dest)
    revision_file = run / 'base_revision.json'
    if revision_file.exists():
        revision = json.loads(revision_file.read_text())['revision']
    else:
        revision = HfApi().model_info('imageomics/bioclip-2').sha
        write_json(revision_file, {'repo': 'imageomics/bioclip-2', 'revision': revision})
    model_dir = snapshot_download('imageomics/bioclip-2', revision=revision,
                                  allow_patterns=['open_clip_config.json', 'open_clip_model.safetensors'])
    base_cfg = json.loads((Path(model_dir) / 'open_clip_config.json').read_text())
    architecture = run / 'ucsc_bioclip.json'
    write_json(architecture, base_cfg['model_cfg'])
    open_clip.add_model_config(architecture)
    preprocess_cfg = base_cfg.get('preprocess_cfg', {})
    model, train_transform, eval_transform = open_clip.create_model_and_transforms(
        'ucsc_bioclip', pretrained=str(Path(model_dir) / 'open_clip_model.safetensors'),
        image_mean=preprocess_cfg.get('mean'), image_std=preprocess_cfg.get('std'),
        image_interpolation=preprocess_cfg.get('interpolation', 'bicubic'),
        image_resize_mode=preprocess_cfg.get('resize_mode', 'shortest'),
        aug_cfg={'scale': (0.7, 1.0)})
    visual = model.visual.cuda().eval()
    del model
    for p in visual.parameters():
        p.requires_grad = False
    dim = int(base_cfg['model_cfg']['embed_dim'])
    head = torch.nn.Linear(dim, nclass).cuda()
    batch = config['batch_size']

    class Photos(Dataset):
        def __init__(self, split, augment=False):
            self.items = [r for r in rows if r['split'] == split]
            self.transform = train_transform if augment else eval_transform

        def __len__(self):
            return len(self.items)

        def __getitem__(self, index):
            row = self.items[index]
            with Image.open(cache / row['path']) as im:
                im = ImageOps.exif_transpose(im).convert('RGB')
                return self.transform(im), row['label']

    def photo_loader(split, augment=False):
        return DataLoader(Photos(split, augment), batch_size=batch, shuffle=augment,
                          num_workers=2, pin_memory=True)

    def embedding_split(split, views=1):
        p = run / f'embeddings_{split}.pt'
        if p.exists():
            data = torch.load(p, map_location='cpu', weights_only=True)
            if data['manifest_hash'] != manifest_hash:
                raise RuntimeError('Stale embedding cache')
            return data['x'], data['y']
        xs, ys = [], []
        with torch.no_grad():
            for view in range(views):
                for x, y in photo_loader(split, augment=(split == 'train')):
                    with torch.autocast('cuda', dtype=torch.float16):
                        features = F.normalize(visual(x.cuda(non_blocking=True)), dim=-1)
                    xs.append(features.cpu().half())
                    ys.append(y)
                print(f'Embedded {split}, view {view + 1}/{views}', flush=True)
        x, y = torch.cat(xs), torch.cat(ys)
        torch.save({'x': x, 'y': y, 'manifest_hash': manifest_hash}, p)
        return x, y

    tx, ty = embedding_split('train', views=2)
    vx, vy = embedding_split('validation')
    head_train = DataLoader(TensorDataset(tx, ty), batch_size=256, shuffle=True)
    head_val = DataLoader(TensorDataset(vx, vy), batch_size=256)
    scaler = torch.amp.GradScaler('cuda')
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
    best = -1.0

    def adapter_state():
        return {name: p.detach().cpu().clone() for name, p in visual.named_parameters() if p.requires_grad}

    def apply_visual(state):
        named = dict(visual.named_parameters())
        with torch.no_grad():
            for name, tensor in state.items():
                named[name].copy_(tensor.to(named[name].device))

    def save_checkpoint(path, optimizer, phase, epoch, stale):
        payload = dict(visual=adapter_state(), head=head.state_dict(), optimizer=optimizer.state_dict(),
                       phase=phase, epoch=epoch, best=best, stale=stale, scaler=scaler.state_dict(),
                       rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(),
                       manifest_hash=manifest_hash, base_revision=revision)
        temp = path.with_suffix('.tmp')
        torch.save(payload, temp)
        temp.replace(path)

    def logits(loader, embeddings=False):
        visual.eval()
        head.eval()
        all_logits, truth = [], []
        with torch.no_grad():
            for x, y in loader:
                with torch.autocast('cuda', dtype=torch.float16):
                    feat = x.cuda().float() if embeddings else F.normalize(visual(x.cuda()), dim=-1)
                    out = head(feat)
                all_logits.append(out.float().cpu())
                truth.append(y)
        return torch.cat(all_logits), torch.cat(truth)

    progress_path = run / 'last.pt'
    resume = torch.load(progress_path, map_location='cpu', weights_only=True) if progress_path.exists() else None
    if resume:
        assert resume['manifest_hash'] == manifest_hash and resume['base_revision'] == revision
        best = resume['best']
    phases = [('head', config['head_epochs']), ('finetune', config['finetune_epochs'])]
    for phase, epochs in phases:
        if phase == 'head' and resume and resume['phase'] == 'finetune':
            continue
        if phase == 'finetune':
            if not (resume and resume['phase'] == phase):
                chosen = torch.load(run / 'best.pt', map_location='cpu', weights_only=True)
                head.load_state_dict(chosen['head'])
            blocks = visual.transformer.resblocks
            for block in blocks[-2:]:
                for p in block.parameters():
                    p.requires_grad = True
            for p in visual.ln_post.parameters():
                p.requires_grad = True
            visual.proj.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': [p for p in visual.parameters() if p.requires_grad], 'lr': 5e-6},
                {'params': head.parameters(), 'lr': 1e-4}], weight_decay=0.01)
            train_loader, val_loader = photo_loader('train', True), photo_loader('validation')
        else:
            optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
            train_loader, val_loader = head_train, head_val
        start, stale = 0, 0
        if resume and resume['phase'] == phase:
            apply_visual(resume['visual'])
            head.load_state_dict(resume['head'])
            optimizer.load_state_dict(resume['optimizer'])
            scaler.load_state_dict(resume['scaler'])
            torch.set_rng_state(resume['rng'])
            torch.cuda.set_rng_state_all(resume['cuda_rng'])
            start, stale = resume['epoch'] + 1, resume['stale']
        for epoch in range(start, epochs):
            if stale >= config['patience']:
                break
            visual.eval()
            head.train()
            total_loss, seen = 0.0, 0
            for step, (x, y) in enumerate(train_loader):
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.float16):
                    feat = x.cuda().float() if phase == 'head' else F.normalize(visual(x.cuda()), dim=-1)
                    loss = loss_fn(head(feat), y.cuda())
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for group in optimizer.param_groups for p in group['params']], 1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += float(loss.detach()) * len(y)
                seen += len(y)
                if phase == 'finetune' and step % 100 == 0:
                    print(f'{phase} epoch {epoch + 1}/{epochs} batch {step}/{len(train_loader)}', flush=True)
            pred, truth = logits(val_loader, embeddings=(phase == 'head'))
            score = float(f1_score(truth.numpy(), pred.argmax(1).numpy(), labels=list(range(nclass)), average='macro', zero_division=0))
            improved = score > best
            stale = 0 if improved else stale + 1
            if improved:
                best = score
                save_checkpoint(run / 'best.pt', optimizer, phase, epoch, stale)
            save_checkpoint(progress_path, optimizer, phase, epoch, stale)
            record = dict(phase=phase, epoch=epoch + 1, loss=total_loss/seen, val_macro_f1=score, best_val_macro_f1=best)
            append_json(run / 'history.jsonl', record)
            print(record, flush=True)
        resume = None

    best_ckpt = torch.load(run / 'best.pt', map_location='cpu', weights_only=True)
    import safetensors.torch
    base_state = safetensors.torch.load_file(str(Path(model_dir) / 'open_clip_model.safetensors'))
    visual.load_state_dict({k.removeprefix('visual.'): v for k, v in base_state.items() if k.startswith('visual.')})
    del base_state
    apply_visual(best_ckpt['visual'])
    head.load_state_dict(best_ckpt['head'])
    cal_logits, cal_y = logits(photo_loader('calibration'))
    temperatures = torch.logspace(-1, 1, 201)
    losses = [float(F.cross_entropy(cal_logits / t, cal_y)) for t in temperatures]
    temperature = float(temperatures[int(np.argmin(losses))])
    test_logits, test_y = logits(photo_loader('test'))
    probability = (test_logits / temperature).softmax(1)
    confidence, prediction = probability.max(1)
    correct = prediction.eq(test_y)
    ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, 16)[:-1], np.linspace(0, 1, 16)[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += float(mask.float().mean() * (correct[mask].float().mean() - confidence[mask].mean()).abs())
    report = classification_report(test_y.numpy(), prediction.numpy(), labels=list(range(nclass)),
                                   target_names=[s['common_name'] for s in classes], output_dict=True, zero_division=0)
    summary = dict(test_top1=float(correct.float().mean()), test_macro_f1=report['macro avg']['f1-score'],
                   test_top3=float((probability.topk(3).indices == test_y[:, None]).any(1).float().mean()),
                   test_ece_15bin=ece, temperature=temperature, test_samples=len(test_y),
                   selected_phase=best_ckpt['phase'], selected_epoch=best_ckpt['epoch'] + 1,
                   best_val_macro_f1=best, base_revision=revision, manifest_sha256=manifest_hash,
                   num_classes=nclass, counts=dict(counts), status='closed_set_research_model')
    write_json(run / 'test_summary.json', summary)
    write_json(run / 'per_class_test.json', report)
    write_json(run / 'confusion_matrix.json', confusion_matrix(test_y, prediction, labels=list(range(nclass))).tolist())
    artifact = dict(format='ucsc-bioclip-v4-adapter', base_repo='imageomics/bioclip-2', base_revision=revision,
                    visual_delta=best_ckpt['visual'], head=best_ckpt['head'], classes=resolved,
                    temperature=temperature, preprocess_cfg=preprocess_cfg, manifest_sha256=manifest_hash)
    torch.save(artifact, run / 'model_adapter.pt')
    write_json(run / 'model_checksum.json', {'sha256': hashlib.sha256((run / 'model_adapter.pt').read_bytes()).hexdigest()})
    print(json.dumps(summary, indent=2), flush=True)
    print(classification_report(test_y.numpy(), prediction.numpy(), labels=list(range(nclass)),
                                target_names=[s['common_name'] for s in classes], zero_division=0), flush=True)


if __name__ == '__main__':
    import json
    main(json.loads(Path(sys.argv[1]).read_text()))

In [ ]:
subprocess.run([sys.executable, '-u', '/content/train_cloud_v4.py',
                '/content/ucsc_v4_config.json'], check=True)
